# 08. 장애인콜택시 대기시간 예측 모델 - Feature Set v2

이 노트북의 목적은 기존 최종 모델에서 사용한 피처 세트를 확장/수정하기 전에, 동일한 데이터 로드와 동일한 train/validation/test 분리 기준을 다시 고정하는 것이다.

이번 노트북에서는 우선 아래까지만 만든다.

1. 기존 데이터 로드
2. train/validation/test 동일하게 분리

예측 대상은 기존과 동일하게 `접수→승차 대기시간`이다.

```text
target_min = 접수_승차_분
```

## 1. 기본 설정

기존 노트북들과 같은 processed CSV를 사용한다.

- 임차택시: `data/processed/임차택시_대기시간_전처리.csv`
- 특장차: `data/processed/특장차_대기시간_전처리_접수유형분류.csv`

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks_waiting_time" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "processed"

RENTAL_PATH = DATA_DIR / "임차택시_대기시간_전처리.csv"
SPECIAL_PATH = DATA_DIR / "특장차_대기시간_전처리_접수유형분류.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RENTAL_PATH:", RENTAL_PATH)
print("SPECIAL_PATH:", SPECIAL_PATH)

## 2. 기존 데이터 로드 함수

모델링 대상은 기존과 동일하게 바로콜 승차완료 건만 사용한다.

- 임차택시: `임차택시_바로콜여부 == True` 그리고 `대기시간분석_포함여부 == True`
- 특장차: `특장차_바로콜_후보여부 == True` 또는 `특장차_접수유형_후보_최종 == "바로콜 후보"`
- 예측 대상: `접수_승차_분`

이번 v2에서는 이후 피처 확장을 위해 원본 컬럼을 넉넉하게 보존한다.

In [ ]:
def existing_cols(path, wanted_cols):
    header = pd.read_csv(path, nrows=0).columns.tolist()
    return [col for col in wanted_cols if col in header]


SEOUL_GU = {
    "강남구", "강동구", "강북구", "강서구", "관악구",
    "광진구", "구로구", "금천구", "노원구", "도봉구",
    "동대문구", "동작구", "마포구", "서대문구", "서초구",
    "성동구", "성북구", "송파구", "양천구", "영등포구",
    "용산구", "은평구", "종로구", "중구", "중랑구",
}


def classify_move_type(row):
    origin = row.get("출발구")
    dest = row.get("목적구")

    origin_in_seoul = origin in SEOUL_GU
    dest_in_seoul = dest in SEOUL_GU

    if origin_in_seoul and dest_in_seoul:
        if origin == dest:
            return "구 내 이동"
        return "구 간 이동"
    if origin_in_seoul and not dest_in_seoul:
        return "서울→서울 외"
    if not origin_in_seoul and dest_in_seoul:
        return "서울 외→서울"
    return "서울 외↔서울 외"


def load_modeling_dataset_v2():
    common_cols = [
        "접수일시", "예정일시", "배차일시", "승차일시", "하차일시", "취소일시",
        "출발구", "출발동", "목적구", "목적동",
        "이용목적", "요금", "승차거리", "차량구분", "장애유형",
        "접수_배차_분", "배차_승차_분", "접수_승차_분",
        "접수_취소_분", "배차_취소_분", "접수승차_날짜차이",
    ]

    rental_cols = common_cols + [
        "예약목적여부",
        "임차택시_바로콜여부",
        "임차택시_장시간예외여부",
        "임차택시_예약성예외여부",
        "임차택시_취소분석유형",
        "대기시간분석_포함여부",
        "대기시간분석_제외사유",
    ]

    special_cols = common_cols + [
        "접수시간대", "접수시간대_HH", "접수요일", "평일주말",
        "세부이동유형", "승차거리_km", "승차거리구간",
        "특장차_접수유형", "특장차_접수유형_분류상태", "특장차_접수유형_메모",
        "접수일자", "예정일자", "취소일자", "접수시", "예정시", "예정시간",
        "취소_접수유형_후보", "_원자료_index",
        "특장차_탑승완료_필수일시존재여부",
        "특장차_탑승완료_시간논리정상여부",
        "심야시간사전예약_후보여부",
        "전일접수_후보여부",
        "특장차_바로콜_후보여부",
        "특장차_접수유형_후보_보완",
        "정기접수_목적후보여부",
        "정기접수_가능패턴여부",
        "동일패턴건수_보완",
        "예정_배차_분", "예정_승차_분",
        "특장차_접수유형_후보_최종",
    ]

    rental = pd.read_csv(
        RENTAL_PATH,
        usecols=existing_cols(RENTAL_PATH, rental_cols),
        low_memory=False,
    )
    special = pd.read_csv(
        SPECIAL_PATH,
        usecols=existing_cols(SPECIAL_PATH, special_cols),
        low_memory=False,
    )

    for frame in [rental, special]:
        for col in [
            "접수일시", "예정일시", "배차일시", "승차일시", "하차일시", "취소일시",
            "접수일자", "예정일자", "취소일자",
        ]:
            if col in frame.columns:
                frame[col] = pd.to_datetime(frame[col], errors="coerce")

    rental_model = rental[
        rental["접수일시"].notna()
        & rental["승차일시"].notna()
        & rental["임차택시_바로콜여부"].fillna(False).astype(bool)
        & rental["대기시간분석_포함여부"].fillna(False).astype(bool)
    ].copy()
    rental_model["model_group"] = "임차택시_바로콜"

    special_model = special[
        special["접수일시"].notna()
        & special["승차일시"].notna()
        & (
            special["특장차_바로콜_후보여부"].fillna(False).astype(bool)
            | special["특장차_접수유형_후보_최종"].eq("바로콜 후보")
        )
    ].copy()
    special_model["model_group"] = "특장차_바로콜"

    all_cols = sorted(set(rental_model.columns) | set(special_model.columns))
    data = pd.concat(
        [
            rental_model.reindex(columns=all_cols),
            special_model.reindex(columns=all_cols),
        ],
        ignore_index=True,
    )

    numeric_cols = [
        "요금", "승차거리", "승차거리_km",
        "접수_배차_분", "배차_승차_분", "접수_승차_분",
        "접수_취소_분", "배차_취소_분", "접수승차_날짜차이",
        "접수시간대_HH", "접수시", "예정시",
        "예정_배차_분", "예정_승차_분", "동일패턴건수_보완",
    ]

    for col in numeric_cols:
        if col in data.columns:
            data[col] = pd.to_numeric(data[col], errors="coerce")

    data = data[
        data["접수일시"].notna()
        & data["승차일시"].notna()
        & data["접수_승차_분"].notna()
        & data["접수_승차_분"].ge(0)
    ].copy()

    data["target_min"] = data["접수_승차_분"]

    # 기존 모델과 동일한 기본 시간 변수
    data["hour"] = data["접수일시"].dt.hour.astype("int16")
    data["dayofweek"] = data["접수일시"].dt.dayofweek.astype("int16")
    data["month"] = data["접수일시"].dt.month.astype("int16")
    data["is_weekend"] = data["dayofweek"].isin([5, 6]).astype("int8")
    data["is_night"] = data["hour"].between(0, 6, inclusive="both").astype("int8")
    data["is_commute"] = (
        data["hour"].between(7, 9, inclusive="both")
        | data["hour"].between(17, 19, inclusive="both")
    ).astype("int8")

    # v2 실험 후보: 02~06시 새벽 위험 시간대
    data["is_dawn_02_06"] = data["hour"].between(2, 6, inclusive="both").astype("int8")

    # 승차거리_km 통일
    if "승차거리_km" not in data.columns:
        data["승차거리_km"] = np.nan
    if "승차거리" in data.columns:
        data["승차거리_km"] = data["승차거리_km"].fillna(
            pd.to_numeric(data["승차거리"], errors="coerce")
        )

    # 세부이동유형 보완
    computed_move_type = data.apply(classify_move_type, axis=1)
    if "세부이동유형" in data.columns:
        data["세부이동유형"] = data["세부이동유형"].fillna(computed_move_type)
        data.loc[data["세부이동유형"].astype(str).eq(""), "세부이동유형"] = computed_move_type
    else:
        data["세부이동유형"] = computed_move_type

    # 문자열 결측 처리
    object_cols = data.select_dtypes(include="object").columns
    for col in object_cols:
        data[col] = data[col].fillna("미상").astype(str)

    data = data.sort_values("접수일시").reset_index(drop=True)

    return data

In [ ]:
data = load_modeling_dataset_v2()

print("data shape:", data.shape)
display(data.head())

display(
    data.groupby("model_group")["target_min"]
    .agg(
        건수="count",
        중앙값="median",
        평균="mean",
        p90=lambda x: x.quantile(0.90),
    )
    .round(2)
)

## 3. train / validation / test 분리

기존 모델링과 동일하게 70% / 15% / 15%로 나눈다.

분리 기준도 기존과 동일하게 `month × model_group`을 stratify 기준으로 사용한다.

이 방식은 월별 분포와 임차택시/특장차 비율이 train/validation/test에 비슷하게 유지되도록 하기 위한 것이다.

In [ ]:
def split_train_valid_test(frame):
    split_key = frame["month"].astype(str) + "_" + frame["model_group"].astype(str)

    train_df, temp_df = train_test_split(
        frame,
        test_size=0.30,
        random_state=RANDOM_STATE,
        stratify=split_key,
    )

    temp_split_key = temp_df["month"].astype(str) + "_" + temp_df["model_group"].astype(str)

    valid_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        random_state=RANDOM_STATE,
        stratify=temp_split_key,
    )

    train_df = train_df.sort_values("접수일시").reset_index(drop=True)
    valid_df = valid_df.sort_values("접수일시").reset_index(drop=True)
    test_df = test_df.sort_values("접수일시").reset_index(drop=True)

    return train_df, valid_df, test_df


train, valid, test = split_train_valid_test(data)

train.shape, valid.shape, test.shape

In [ ]:
def split_summary(frame, name):
    return {
        "split": name,
        "rows": len(frame),
        "target_mean": frame["target_min"].mean(),
        "target_median": frame["target_min"].median(),
        "target_p90": frame["target_min"].quantile(0.90),
        "min_date": frame["접수일시"].min(),
        "max_date": frame["접수일시"].max(),
    }

split_summary_df = pd.DataFrame([
    split_summary(train, "train"),
    split_summary(valid, "valid"),
    split_summary(test, "test"),
])

display(split_summary_df.round(4))

print("model_group 비율")
display(
    pd.concat(
        [
            train["model_group"].value_counts(normalize=True).rename("train"),
            valid["model_group"].value_counts(normalize=True).rename("valid"),
            test["model_group"].value_counts(normalize=True).rename("test"),
        ],
        axis=1,
    ).round(4)
)

print("month 비율")
display(
    pd.concat(
        [
            train["month"].value_counts(normalize=True).sort_index().rename("train"),
            valid["month"].value_counts(normalize=True).sort_index().rename("valid"),
            test["month"].value_counts(normalize=True).sort_index().rename("test"),
        ],
        axis=1,
    ).round(4)
)

## 4. 다음 단계

다음 셀부터는 새 피처 세트를 하나씩 추가하면서 기존 최종 RF 성능과 비교한다.

기존 최종 RF 기준 성능:

```text
MAE: 15.06
RMSE: 23.51
R²: 0.573
Recall: 0.421
F1: 0.548
```